[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/fitting/fitting.ipynb)

# Fitting Points, Lines and Planes

This example fits a point, a line, and a plane to noisy point clouds and triangulates the point closest to a bundle of lines with one function. Textbook treatments split these four tasks into separate recipes for a centroid, a principal axis, a plane normal of least spread, and a linear projection system for skew rays. Here all four minimize the same squared join residual.

Joining a normalized sample with an unknown primitive measures their Euclidean separation: `samples & Unknown` produces a signed distance scalar for a point and a plane, a spanned plane whose norm is the perpendicular distance for a point and a line, and a joining line whose norm is the point-to-point distance for two points. Because `samples & Unknown` is linear in `Unknown`, summing `residual.scalar_norm_squared()` builds one quadratic form on the unknown's type, and `.eigh()` extracts its flattest eigenvector normalized in the type's degenerate reverse metric, leaving position components free while fixing geometric weight, direction, or normal norm. Because `&` treats points and lines symmetrically, passing `Point` with a bundle of lines triangulates their point of closest approach without changing a line of code.

In matrix least-squares notation, the four fits split into the sample mean $\bar{\mathbf x} = \frac{1}{N}\sum_k \mathbf x_k$, the leading right singular vector of the centred data matrix $X - \mathbf 1\bar{\mathbf x}^{\mathsf T}$ paired with $\bar{\mathbf x}$ as an anchor point, the trailing right singular vector $\mathbf n$ with offset $d = \mathbf n^{\mathsf T}\bar{\mathbf x}$, and the normal equations $\bigl(\sum_k (I - \mathbf d_k \mathbf d_k^{\mathsf T})\bigr)\mathbf x = \sum_k (I - \mathbf d_k \mathbf d_k^{\mathsf T})\mathbf p_k$ for unit rays through $\mathbf p_k$ along $\mathbf d_k$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, GAType, NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.geometry.fitting import render
from examples.geometry.fitting.scenarios import (
    line_segment,
    noisy_points,
    planar_patch,
    point_cloud,
    ray_bundle,
    segment_end_planes,
    square_patch_edges,
)
from examples.util.animation import save_figure

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                  # Antivector-
# A line through points is their join, intrinsic like them.
Line = ga.gatype.antibivector()                 # Antibivector-
Plane = ga.gatype.vector()                      # Vector+
origin = mv.w.dual()                            # Antivector-

# Shared rigid pose and sample noise used across the scenes.
shift, tilt, spin = np.array([0.4, -0.3, 0.6]), 0.3, 0.5
pose_shift = ((stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0)).exp()
pose = pose_shift * (mv.yz * tilt).exp() * (mv.xy * spin).exp()                 # [] Motor+
seed, noise_sigma = 0, 0.05

## Fitting a point to a cloud

Passing `Point` as the identity map on unknown points evaluates every sample's join with an arbitrary point and sums the squared distances into one quadratic form. Minimizing that form under unit point weight recovers the cloud's centroid.

In [ ]:
def fitted_primitive(Unknown: GAType, samples: Extensor) -> Extensor:
    """The primitive of type `Unknown` fitted to `samples` (`[n_samples]`) is the flattest direction of their summed squared join residual."""
    samples = samples.normalized()
    # Joining each sample with the unknown measures its incidence error:
    # point & plane is a scalar, point & line a plane, point & point a line.
    residual = samples & Unknown
    misfit = residual.scalar_norm_squared().sum(axis=0)

    # The unknown's own metric, the scalar part of the primitive's reverse product, fixes its
    # geometric size, leaving its position free: plane normal, line direction, or point weight
    # has unit norm. For a line it leaves out the condition to be a line, the pseudoscalar part;
    # the fitted moment is the centroid's moment about the fitted direction, which meets that
    # condition. The metric is blind to the ideal components, so for each mode eigh chooses them
    # to minimize the misfit:
    eigenvalues, modes = misfit.eigh()
    return modes[eigenvalues.argmin()]


n_cloud, cloud_spread = 200, 0.5
rng = np.random.default_rng(seed)
raw_cloud = point_cloud(n_cloud, cloud_spread, rng)                             # [n_cloud] Point-
base_cloud = pose >> raw_cloud                                                  # [n_cloud] Point-
cloud_points = noisy_points(base_cloud, noise_sigma, rng)                       # [n_cloud] Point-
centroid = fitted_primitive(Point, cloud_points)                                # [] Point-
true_centre = pose >> origin                                                    # [] Point-
save_figure(render.point_fit_figure(cloud_points, true_centre, centroid), "fitting_point_to_points");

Blue spheres mark the noisy sample cloud, the grey sphere marks the true centre before sampling, and the red sphere marks the fitted centroid.

## Fitting a line to a segment

Joined with an unknown `Line`, each sample point yields the plane through both, whose norm equals the perpendicular distance from the point to the line. Although `Line` spans all six antibivectors, the moment components of the minimal mode automatically satisfy the Plücker line condition because they equal the centroid's moment about the fitted unit direction.

In [ ]:
n_segment, half_length, drawn_half_length = 120, 2.0, 2.5
rng = np.random.default_rng(seed)
raw_segment = line_segment(n_segment, half_length)                              # [n_segment] Point-
base_segment = pose >> raw_segment                                              # [n_segment] Point-
segment_points = noisy_points(base_segment, noise_sigma, rng)                   # [n_segment] Point-
fitted_line = fitted_primitive(Line, segment_points)                            # [] Line-
y_axis_line = origin & mv.direction([0, 1, 0])                                  # [] Line-
true_line = pose >> y_axis_line                                                 # [] Line-
local_caps = segment_end_planes(drawn_half_length)                              # [2] Plane+
end_caps = pose >> local_caps                                                   # [2] Plane+
fitted_line_ends = fitted_line ^ end_caps                                       # [2] Point-
save_figure(render.line_fit_figure(segment_points, true_line ^ end_caps, fitted_line_ends), "fitting_line_to_points");

Blue spheres show the perturbed segment samples, while the grey rod and red rod show the true axis and the fitted line clipped between the same two cap planes.

## Fitting a plane to a patch

Joined with an unknown `Plane`, each normalized sample point gives its signed orthogonal distance to the plane. The degenerate reverse norm on `Plane` constrains the normal vector to unit length while leaving the plane's offset free to pass through the sample centroid.

In [ ]:
n_patch, half_width = 200, 2.0
rng = np.random.default_rng(seed)
raw_patch = planar_patch(n_patch, half_width, rng)                              # [n_patch] Point-
base_patch = pose >> raw_patch                                                  # [n_patch] Point-
patch_points = noisy_points(base_patch, noise_sigma, rng)                       # [n_patch] Point-
fitted_plane = fitted_primitive(Plane, patch_points)                            # [] Plane+
true_plane = pose >> mv.z                                                       # [] Plane+
local_edges = square_patch_edges(half_width)                                    # [4] Line-
frame_edges = pose >> local_edges                                               # [4] Line-
fitted_quad = fitted_plane ^ frame_edges                                        # [4] Point-
save_figure(render.plane_fit_figure(patch_points, true_plane ^ frame_edges, fitted_quad), "fitting_plane_to_points");

Blue spheres mark the noisy planar patch, and the grey and red wire quads show where four boundary lines cut the true plane and the fitted plane.

## Triangulating a point from a bundle of lines

When the samples are skew lines instead of points, joining each line with an unknown `Point` again produces the plane through both, whose norm is the point's perpendicular distance to that ray. Minimizing the summed squared norm triangulates the point of closest approach.

In [ ]:
n_rays, ray_spread, ray_half_length = 30, 0.05, 2.0
rng = np.random.default_rng(seed)
local_rays = ray_bundle(n_rays, ray_spread, rng)                                # [n_rays] Line-
rays = pose >> local_rays                                                       # [n_rays] Line-
intersection = fitted_primitive(Point, rays).normalized()                       # [] Point-
# Each line's point at infinity is its direction; its length is the norm of its complement.
directions = rays ^ mv.w                                                        # [n_rays] Point-
unit_directions = directions / directions.dual().norm()                         # [n_rays] Point-
ray_ends = intersection + unit_directions * np.array([[-ray_half_length], [ray_half_length]])   # [2, n_rays] Point-
save_figure(render.bundle_fit_figure(ray_ends, true_centre, intersection), "fitting_point_to_lines");

Blue rods trace the skew rays of the bundle, the grey sphere marks the true bundle centre, and the red sphere marks the triangulated point of closest approach.

In [ ]:
# checks
# The fitted antibivector satisfies the Plücker line condition (its self-join vanishes),
# and finite sample noise keeps the fitted primitives within centimetres of the posed truth.
assert (fitted_line & fitted_line).abs() < 1e-10
assert (centroid.normalized() & true_centre.normalized()).norm() < 0.1
assert ((fitted_line_ends.normalized() & true_line.normalized()).norm() < 0.05).all()
assert ((true_plane.normalized() & fitted_quad.normalized()).norm() < 0.05).all()
assert (intersection & true_centre.normalized()).norm() < 0.05